In [28]:
import pandas as pd
import numpy as np
import re
import urllib.parse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. DATA LOADING ---
def load_datasets():
    try:
        # Load datasets with error handling for bad lines
        books = pd.read_csv('books.csv', on_bad_lines='warn')
        articles = pd.read_csv('medium_data.csv', on_bad_lines='warn')

        print("Datasets loaded successfully.")
        return books, articles
    except Exception as e:
        print(f"Error: {e}")
        return None, None

books_df, articles_df = load_datasets()

Datasets loaded successfully.


/tmp/ipykernel_400/1449199444.py:12: ParserWarning: Skipping line 3350: expected 12 fields, saw 13
Skipping line 4704: expected 12 fields, saw 13
Skipping line 5879: expected 12 fields, saw 13
Skipping line 8981: expected 12 fields, saw 13

  books = pd.read_csv('books.csv', on_bad_lines='warn')


In [29]:
# --- 2. DATA PREPROCESSING & FEATURE ENGINEERING ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    return text.strip()

# Combine multiple fields for richer recommendation context
books_df['metadata'] = (
    books_df['title'].fillna('') + " " +
    books_df['authors'].fillna('') + " " +
    books_df.get('categories', pd.Series(['']*len(books_df))).fillna('')
).apply(clean_text)

articles_df['metadata'] = (
    articles_df['title'].fillna('') + " " +
    articles_df.get('subtitle', pd.Series(['']*len(articles_df))).fillna('')
).apply(clean_text)

print("Preprocessing complete: Metadata fields generated.")

Preprocessing complete: Metadata fields generated.


In [30]:
# --- 3. RECOMMENDATION ENGINE & NLP EXTRACTION ---
def extract_keywords(user_input):
    """Extracts multiple interests from a natural language sentence."""
    text = user_input.lower()
    # Remove common filler phrases
    fillers = ['i want to learn', 'i am interested in', 'show me', 'about', 'and', 'the']
    for word in fillers:
        text = text.replace(word, ',')

    # Split by comma and clean
    interests = [i.strip() for i in text.split(',') if len(i.strip()) > 1]
    return list(set(interests)) # Remove duplicates

def get_recommendations(query, df, top_n=5):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df['metadata'])
    query_vec = tfidf.transform([clean_text(query)])

    # Calculate Similarity
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    indices = np.argsort(similarity)[::-1][:top_n]
    return df.iloc[indices]

def make_yt_link(topic):
    return f"https://www.youtube.com/results?search_query={urllib.parse.quote(topic + ' tutorial')}"

In [32]:
# --- 4. LEARNING PATH GENERATOR ---
def get_structured_path(interest):
    paths = {
        "ai": ["Python Programming", "Math (Linear Algebra & Calculus)", "Machine Learning Theory", "Deep Learning (Neural Networks)", "Portfolio Project (Deployment)"],
        "robotics": ["Intro to ROS", "C++ Fundamentals", "Sensors & Actuators", "Computer Vision", "Robot Kinematics"],
        "programming": ["Computational Logic", "Data Structures", "Algorithm Design", "API Development", "Full Stack Capstone"]
    }
    # Default path for unknown interests
    return paths.get(interest.lower(), ["Fundamentals", "Intermediate Concepts", "Advanced Tools", "Optimization", "Real-world Project"])

In [34]:
# --- 5. USER INTERACTION & DISPLAY ---
user_query = input("Hello! I am your AI Learning Assistant. What would you like to learn today?\n")
interests = extract_keywords(user_query)

for topic in interests:
    print(f"\n{'*'*60}")
    print(f"RESULTS FOR: {topic.upper()}")
    print(f"{'*'*60}")

    print("\n📚 Recommended Books:")
    for _, row in get_recommendations(topic, books_df).iterrows():
        print(f"- {row['title']} by {row.get('authors', 'Unknown')}")

    print("\n📰 Top Articles:")
    for _, row in get_recommendations(topic, articles_df).iterrows():
        print(f"- {row['title']}")

    print("\n🎥 YouTube Learning Link:")
    print(f"  {make_yt_link(topic)}")

    print("\n🚀 Your Structured Learning Path:")
    for i, step in enumerate(get_structured_path(topic), 1):
        print(f"  Step {i}: {step}")

Hello! I am your AI Learning Assistant. What would you like to learn today?
Cooking

************************************************************
RESULTS FOR: COOKING
************************************************************

📚 Recommended Books:
- More Classic Italian Cooking by Marcella Hazan
- Julie and Julia: My Year of Cooking Dangerously by Julie Powell
- How to Be a Domestic Goddess: Baking and the Art of Comfort Cooking by Nigella Lawson
- Essentials of Classic Italian Cooking by Marcella Hazan/Karin Kretschmann
- How to Be a Domestic Goddess: Baking and the Art of Comfort Cooking by Nigella Lawson/Petrina Tinslay

📰 Top Articles:
- <strong class="markup--strong markup--h3-strong">The future of cooking — why it is beyond the screen</strong>
- Reality is not always what it appears to be
- Want to write better? Try cooking
- Topics in Online-News before the Austrian Elections 2019
- Set for Life

🎥 YouTube Learning Link:
  https://www.youtube.com/results?search_query=cooking%2

### 🛠 How to Run Your Streamlit App

1. **Install Streamlit**:
   ```bash
   pip install streamlit
   ```

2. **Save the code**: Copy the code below into a file named `app.py` in the same folder as your CSV files.

3. **Run the App**:
   ```bash
   streamlit run app.py
   ```

*Note: In Google Colab, you would typically use `localtunnel` or a similar tool to view a Streamlit app, but for a portfolio project, this code is ready for local use or deployment to Streamlit Cloud.*

### 🚀 Portfolio & Deployment Strategy

#### 1. GitHub Structure
```text
AI-Learning-Assistant/
├── data/               # CSV files
├── notebooks/          # This Colab Notebook
├── app.py              # Future Streamlit script
├── requirements.txt    # pandas, scikit-learn, streamlit
└── README.md           # Project documentation
```

#### 2. README Description
**AI Learning Assistant**  
*An intelligent resource discovery engine using NLP and Content-Based Filtering. This system parses user intent to recommend multi-format educational content (Books, Articles, Videos) and generates personalized learning roadmaps using TF-IDF and Cosine Similarity.*

#### 3. Converting to Streamlit
To turn this into a web app:
1.  **Install Streamlit**: `pip install streamlit`.
2.  **Move Logic**: Put the `get_recommendations` and `clean_text` functions into an `app.py` file.
3.  **Create UI**: Use `st.text_input()` for the query and `st.write()` to display the results in columns or tabs.
4.  **Deploy**: Use Streamlit Cloud to host the app directly from your GitHub repo.

In [39]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import re
import urllib.parse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- PAGE CONFIG ---
st.set_page_config(page_title="AI Learning Assistant", page_icon="🎓", layout="wide")

# --- STYLES ---
st.markdown("""
    <style>
    .main { background-color: #f5f7f9; }
    .stMetric { background-color: #ffffff; padding: 10px; border-radius: 10px; border: 1px solid #e0e0e0; }
    </style>
""", unsafe_allow_html=True)

# --- CACHED DATA LOADING & PROCESSING ---
@st.cache_data
def load_and_preprocess():
    books = pd.read_csv('books.csv', on_bad_lines='warn')
    articles = pd.read_csv('medium_data.csv', on_bad_lines='warn')

    def clean(text):
        return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

    # Feature Engineering: Combine multiple fields for deeper context
    books['metadata'] = (books['title'].fillna('') + " " +
                         books['authors'].fillna('') + " " +
                         books.get('categories', pd.Series(['']*len(books))).fillna('')).apply(clean)

    articles['metadata'] = (articles['title'].fillna('') + " " +
                            articles.get('subtitle', pd.Series(['']*len(articles))).fillna('')).apply(clean)
    return books, articles

@st.cache_resource
def compute_tfidf(metadata_series):
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(metadata_series)
    return vectorizer, matrix

# --- LOGIC ---
def get_recommendations(query, df, vectorizer, matrix, top_n=5):
    query_vec = vectorizer.transform([query.lower()])
    similarity = cosine_similarity(query_vec, matrix).flatten()
    indices = np.argsort(similarity)[::-1][:top_n]

    results = df.iloc[indices].copy()
    results['score'] = similarity[indices]
    return results

def get_learning_path(topic):
    paths = {
        "ai": ["Python for DS", "Linear Algebra & Calculus", "Supervised Learning", "Neural Networks", "LLM Fine-tuning"],
        "machine learning": ["Statistics & Probability", "Data Cleaning", "Regression & Classification", "Ensemble Methods", "Model Deployment"],
        "data science": ["SQL & Databases", "Exploratory Data Analysis", "Statistical Modeling", "Data Visualization", "Business Analytics"],
        "physics": ["Classical Mechanics", "Electromagnetism", "Thermodynamics", "Quantum Mechanics", "Special Relativity"],
        "robotics": ["C++ Programming", "Kinematics", "Sensor Fusion", "Path Planning", "Embedded Systems"],
        "programming": ["Logic", "Memory Management", "Design Patterns", "System Architecture", "Cloud Infrastructure"]
    }
    return paths.get(topic.lower(), ["Fundamentals", "Foundational Tools", "Advanced Concepts", "Practical Application", "Specialized Project"])

# --- SIDEBAR ---
with st.sidebar:
    st.title("🛠 How it Works")
    st.info("""
    **1. NLP Preprocessing**
    Cleaning and tokenizing user input.

    **2. TF-IDF Vectorization**
    Converting text into mathematical vectors based on term importance.

    **3. Cosine Similarity**
    Measuring the angle between user query and resource metadata.
    """)
    st.markdown("--- ")
    st.write("**Try Example Topics:**")
    if st.button("🤖 AI"): st.session_state.topic = "AI"
    if st.button("🧪 Physics"): st.session_state.topic = "Physics"
    if st.button("📊 Data Science"): st.session_state.topic = "Data Science"

# --- MAIN UI ---
st.title("🤖 AI Learning Assistant")
st.write("Enter your interest to generate a professional learning roadmap.")

if 'topic' not in st.session_state: st.session_state.topic = ""

user_input = st.text_input("What would you like to master?", value=st.session_state.topic, placeholder="e.g. Computer Vision")

if st.button("Generate Learning Plan"):
    if not user_input:
        st.warning("⚠️ Please enter a topic to continue.")
    else:
        with st.spinner(f"Analyzing resources for '{user_input}'..."):
            books_df, articles_df = load_and_preprocess()

            # Load/Compute Vectors
            b_vec, b_mat = compute_tfidf(books_df['metadata'])
            a_vec, a_mat = compute_tfidf(articles_df['metadata'])

            rec_books = get_recommendations(user_input, books_df, b_vec, b_mat)
            rec_articles = get_recommendations(user_input, articles_df, a_vec, a_mat)

            st.success("Roadmap Generated!")

            col1, col2 = st.columns([2, 1])

            with col1:
                st.subheader("📚 Top Recommended Books")
                for _, row in rec_books.iterrows():
                    with st.expander(f"{row['title']}"):
                        st.write(f"**Author:** {row.get('authors', 'Unknown')}")
                        st.metric("Match Score", f"{row['score']*100:.1f}%")

                st.subheader("📰 Insightful Articles")
                for _, row in rec_articles.iterrows():
                    st.markdown(f"- {row['title']} `({row['score']*100:.1f}% Match)`")

            with col2:
                st.subheader("🚀 Learning Path")
                path = get_learning_path(user_input)
                for i, step in enumerate(path, 1):
                    st.info(f"**Step {i}**: {step}")

                st.subheader("🎥 Video Content")
                yt_url = f"https://www.youtube.com/results?search_query={urllib.parse.quote(user_input + ' tutorial')}"
                st.link_button("Search YouTube Tutorials", yt_url, use_container_width=True)
"

Overwriting app.py


### 🚀 Deployment Roadmap (Streamlit Community Cloud)

#### 1. Prepare your GitHub Repository
Ensure your folder looks exactly like this:
```text
portfolio-recommender/
├── app.py
├── books.csv
├── medium_data.csv
└── requirements.txt
```

#### 2. Create requirements.txt
Create a file named `requirements.txt` with these contents:
```text
streamlit
pandas
numpy
scikit-learn
```

#### 3. Deploy
1. Go to [share.streamlit.io](https://share.streamlit.io/).
2. Connect your GitHub account.
3. Select your repository and the `app.py` file.
4. Click **Deploy**! Your app will be live in minutes.

### Portfolio Organization Guide

#### 1. GitHub Repository Structure:
```text
AI-Learning-Recommender/
├── data/
│   ├── books.csv
│   ├── youtube.csv
│   └── medium_data.csv
├── notebooks/
│   └── recommendation_system.ipynb
├── src/
│   └── engine.py (optional: extract functions here)
├── README.md
└── requirements.txt
```

#### 2. README Description:
**Project Title: AI Learning Resource Recommender**
> An NLP-powered recommendation engine that generates personalized learning paths and resources (Books, Articles, Videos) based on user interests. Built using Scikit-Learn's TF-IDF Vectorization and Cosine Similarity to match user intent with multi-dataset metadata.